# Invoke cost reporting: max -> progress -> actual

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/cost_reporting.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/cost_reporting.ipynb): it reads only a small shard-map fixture checked into the git tree -- no cloud data, no credentials. The one step Binder cannot do (invoke AWS Lambda) is replayed with a stand-in, clearly marked below._

A zagg Lambda fan-out spends real money, so every run reports three cost figures (issue #298):

| figure | when | meaning |
|---|---|---|
| **max** | before any invoke | hard ceiling: `n_units x rate(arch) x memory_gb x 900 s` -- every unit is one Lambda invocation billed at the configured memory for at most the function timeout |
| **estimated** | before any invoke | prior-run history estimate (pilot-first); currently a `None` placeholder -- deferred behind the stats-sidecar and template-hash issues |
| **actual** | after the run | rollup of billed durations: `sum(duration_s) x memory_gb x rate` |

The ceiling is exact and cheap: it needs only the shard map (how many units) and the config (which worker memory).

## Max cost, before any invoke

`zagg.notebook.max_cost_preview` resolves the ceiling from a shard map + config with the same unit accounting the dispatcher uses (cell selection, windowed-unit expansion). The fixture below is a 4-shard ATL03 shard map from the test tree.

In [ ]:
from pathlib import Path

from zagg.config import default_config
from zagg.notebook import format_max_cost, max_cost_preview


def find(rel):
    """Resolve a repo file whether we run from the repo root or notebooks/."""
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / rel).exists():
            return str(base / rel)
    raise FileNotFoundError(rel)


shardmap = find("tests/data/benchmark/shardmaps/sm_healpix_o9.json")
config = default_config("atl06")

preview = max_cost_preview(config, shardmap)
print(format_max_cost(preview))
preview

The ceiling scales with the worker-size variant the config selects (the `worker:` block, issue #235):

In [ ]:
for memory_mb in (2048, 4096, 8192):
    config.worker = {"memory": memory_mb}
    p = max_cost_preview(config, shardmap)
    print(f"worker {memory_mb} MB -> ceiling ${p['max_cost_usd']:.4f}")
config.worker = None  # back to the default 4 GB worker

## Estimated cost (deferred)

The estimator is an interface stub for now: **tier 1 (pilot-first)** scales prior actuals for the *same* template hash by per-shard granule/obs counts (the dev -> single-shard test -> fleet workflow means a pilot run usually exists); **tier 2** falls back to a cross-template regression on per-shard `n_obs`. Both need the per-shard stats sidecars and template-hash identity to exist first, so until those land the stub returns `None` and `cost["estimated_cost_usd"]` stays a placeholder.

In [ ]:
from zagg.dispatch import estimate_cost_usd

print(estimate_cost_usd())  # None until the sidecar history exists

## Progress + report

`zagg.notebook.run` wraps `zagg.agg` with a per-unit progress bar (tqdm when importable, logging otherwise; the running cost rides the postfix) and returns a `RunView` -- a summary dict with a rich HTML repr. The notebook path is **informational only**: it displays the ceiling and never prompts (the blocking yes/no gate is CLI-only).

Binder cannot invoke AWS Lambda, so the cell below substitutes a stand-in `agg` that replays a realistic run summary while ticking the same `on_progress` callback a real Lambda fan-out drives -- the wrapper code path (ceiling display, bar, `RunView`) is exactly the one a real run exercises.

In [ ]:
import time
from unittest.mock import patch

from zagg.dispatch import LAMBDA_PRICE_PER_GB_SEC

MEMORY_GB = 4.0
DURATIONS = {-231928: 41.0, -231921: 63.0, -231920: 55.0, -231913: 48.0}  # simulated billed seconds


def replay_agg(config, *, on_progress=None, **kwargs):
    """Stand-in for zagg.agg: replays a recorded-style Lambda run summary."""
    total, cost = len(DURATIONS), 0.0
    for i, duration in enumerate(DURATIONS.values(), 1):
        time.sleep(0.3)  # a real shard takes seconds-to-minutes
        cost += duration * MEMORY_GB * LAMBDA_PRICE_PER_GB_SEC
        if on_progress is not None:
            on_progress(i, total, cost)
    lambda_time = sum(DURATIONS.values())
    actual = lambda_time * MEMORY_GB * LAMBDA_PRICE_PER_GB_SEC
    return {
        "backend": "lambda",
        "store_path": "s3://example-bucket/atl03_serc.zarr",
        "total_cells": total,
        "cells_with_data": total,
        "cells_error": 0,
        "total_obs": 1_284_055,
        "wall_time_s": 71.2,
        "lambda_time_s": lambda_time,
        "cost": {
            "max_cost_usd": preview["max_cost_usd"],
            "estimated_cost_usd": None,
            "actual_cost_usd": actual,
        },
        "results": [
            {"shard_key": k, "status_code": 200, "error": None, "lambda_duration": d}
            for k, d in DURATIONS.items()
        ],
    }


from zagg import notebook

with patch("zagg.runner.agg", replay_agg):
    report = notebook.run(config, catalog=shardmap, backend="lambda")

In [ ]:
report  # RunView: cost block, counters, failures render as HTML

In [ ]:
cost = report["cost"]
assert cost["max_cost_usd"] >= cost["actual_cost_usd"]  # the ceiling always bounds the bill
cost

## Running it for real

Against a deployed fleet the call is the same, minus the stand-in:

```python
from zagg import load_config, notebook

config = load_config("atl03_serc.yaml")
report = notebook.run(
    config,
    catalog="shardmap_atl03_serc.json",
    backend="lambda",
    store="s3://your-bucket/atl03_serc.zarr",
)
report  # max / estimated / actual + per-shard failures
```

From the command line the same run **blocks on the ceiling** with a yes/no prompt; `--yes`/`-y` skips it for scripted runs:

```console
$ python -m zagg --config atl03_serc.yaml --catalog shardmap_atl03_serc.json --backend lambda
Max cost ceiling: ~$0.19 (4 units x 4 GB x 900s, arm64)
Proceed with Lambda fan-out? [y/N] y
```